In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:
def lattice_spacing_su2(beta: float) -> float:
    """SU(2) lattice spacing from arXiv:1811.02800."""
    t0_phys = 0.01133  # fm^2
    delta_beta = beta - 2.600
    ln_t0_over_a2 = 1.285 + 6.409 * delta_beta - 0.7411 * delta_beta**2
    t0_over_a2 = np.exp(ln_t0_over_a2)
    return np.sqrt(t0_phys / t0_over_a2)


def lattice_spacing_su3(beta: float) -> float:
    """SU(3) lattice spacing from arXiv:hep-lat/0108008."""
    r0 = 0.5  # fm
    delta_beta = beta - 6.0
    ln_a_over_r0 = -1.6804 - 1.7331 * delta_beta + 0.7849 * delta_beta**2 - 0.4428 * delta_beta**3
    return np.exp(ln_a_over_r0) * r0


In [4]:
# ---------------------------------------------------------------------------
# Volume-preserving lattice adjustment
# ---------------------------------------------------------------------------

def adjust_lattice(T_ref: int, L_ref: int, a_ref_fm: float, a_new_fm: float,
                   open_bc: bool = False, n_exclude_ref: int = 2) -> dict:
    """Compute (T_new, L_new) keeping physical volume constant.

    For open BC, n_exclude scales with lattice spacing to keep the
    physical exclusion distance d_phys = n_exclude_ref * a_ref constant:
        n_exclude_new = round(n_exclude_ref * a_ref / a_new)
    T_new is then adjusted so that T_eff * a stays constant.
    """
    ratio = a_ref_fm / a_new_fm
    L_new = round(L_ref * ratio)

    if open_bc:
        # Physical exclusion distance fixed from reference
        d_phys = n_exclude_ref * a_ref_fm
        n_exclude_new = max(1, round(d_phys / a_new_fm))

        T_eff_ref = T_ref - 2 * n_exclude_ref
        T_new = round(T_eff_ref * ratio) + 2 * n_exclude_new
        T_eff_new = T_new - 2 * n_exclude_new
    else:
        n_exclude_new = 0
        T_new = round(T_ref * ratio)
        T_eff_new = T_new

    V_fm4 = T_eff_new * L_new**3 * a_new_fm**4
    return {"T_new": T_new, "L_new": L_new, "T_eff_new": T_eff_new,
            "n_exclude": n_exclude_new, "V_fm4": V_fm4}

In [5]:
# ---------------------------------------------------------------------------
# Scan: beta=2.5 upward, reference T=17, L=13, open BC  [SU(2)]
# ---------------------------------------------------------------------------

T_ref, L_ref = 160, 80
beta_ref  = 3.15
a_ref_fm  = lattice_spacing_su2(beta_ref)
open_bc   = True
n_exclude_ref = 40

beta_values = np.arange(3.15, 3.25, 0.05)

d_phys = n_exclude_ref * a_ref_fm
print(f"Reference: beta={beta_ref}, T={T_ref}, L={L_ref}, a={a_ref_fm:.4f} fm, open_bc={open_bc}")
print(f"Physical exclusion distance: {d_phys:.4f} fm  (n_exclude_ref={n_exclude_ref})")
print()
print(f"{'beta':>8} {'a [fm]':>10} {'T_new':>7} {'L_new':>7} {'T_eff':>7} {'n_excl':>7} {'V [fm^4]':>12}")
print("-" * 65)

for beta in beta_values:
    a = lattice_spacing_su2(beta)
    adj = adjust_lattice(T_ref, L_ref, a_ref_fm, a, open_bc=open_bc, n_exclude_ref=n_exclude_ref)
    print(f"{beta:8.2f} {a:10.4f} {adj['T_new']:7d} {adj['L_new']:7d} "
          f"{adj['T_eff_new']:7d} {adj['n_exclude']:7d} {adj['V_fm4']:12.6f}")

Reference: beta=3.15, T=160, L=80, a=0.0107 fm, open_bc=True
Physical exclusion distance: 0.4299 fm  (n_exclude_ref=40)

    beta     a [fm]   T_new   L_new   T_eff  n_excl     V [fm^4]
-----------------------------------------------------------------
    3.15     0.0107     160      80      80      40     0.546604
    3.20     0.0094     184      92      92      46     0.548452
    3.25     0.0082     211     105     105      53     0.537820


In [6]:
# ---------------------------------------------------------------------------
# Scan: beta=6.1 upward, reference T=18, L=14, open BC  [SU(3)]
# ---------------------------------------------------------------------------

T_ref, L_ref = 60, 30
beta_ref  = 6.594
a_ref_fm  = lattice_spacing_su3(beta_ref)
open_bc   = True
n_exclude_ref = 0

beta_values = np.arange(6.1, 7.45, 0.05)

d_phys = n_exclude_ref * a_ref_fm
print(f"Reference: beta={beta_ref}, T={T_ref}, L={L_ref}, a={a_ref_fm:.4f} fm, open_bc={open_bc}")
print(f"Physical exclusion distance: {d_phys:.4f} fm  (n_exclude_ref={n_exclude_ref})")
print()
print(f"{'beta':>8} {'a [fm]':>10} {'T_new':>7} {'L_new':>7} {'T_eff':>7} {'n_excl':>7} {'V [fm^4]':>12}")
print("-" * 65)

for beta in beta_values:
    a = lattice_spacing_su3(beta)
    adj = adjust_lattice(T_ref, L_ref, a_ref_fm, a, open_bc=open_bc, n_exclude_ref=n_exclude_ref)
    print(f"{beta:8.2f} {a:10.4f} {adj['T_new']:7d} {adj['L_new']:7d} "
          f"{adj['T_eff_new']:7d} {adj['n_exclude']:7d} {adj['V_fm4']:12.6f}")

Reference: beta=6.594, T=60, L=30, a=0.0400 fm, open_bc=True
Physical exclusion distance: 0.0000 fm  (n_exclude_ref=0)

    beta     a [fm]   T_new   L_new   T_eff  n_excl     V [fm^4]
-----------------------------------------------------------------
    6.10     0.0789      32      15      30       1     3.925698
    6.15     0.0730      35      16      33       1     3.837757
    6.20     0.0677      37      18      35       1     4.293958
    6.25     0.0630      40      19      38       1     4.104830
    6.30     0.0587      43      20      41       1     3.902439
    6.35     0.0549      46      22      44       1     4.243757
    6.40     0.0513      49      23      47       1     3.968788
    6.45     0.0481      52      25      50       1     4.175567
    6.50     0.0451      55      27      53       1     4.309896
    6.55     0.0423      59      28      57       1     4.005633
    6.60     0.0397      62      30      60       1     4.022827
    6.65     0.0373      66      3

In [7]:
# ---------------------------------------------------------------------------
# Corrected: Expected spread for a FIXED lattice size
# ---------------------------------------------------------------------------

HBAR_C = 197.3  # MeV·fm

# Target susceptibility scales
chi_scales = {
    "SU(2)": 200.0, # MeV
    "SU(3)": 191.0  # MeV
}

# Reference geometries
geometries = {
    "SU(2)": {"T": 20, "L": 16, "beta": 2.5, "func": lattice_spacing_su2},
    "SU(3)": {"T": 20, "L": 16, "beta": 6.1, "func": lattice_spacing_su3}
}

for label in ["SU(2)", "SU(3)"]:
    # 1. Get parameters for this group
    geom = geometries[label]
    a_fm = geom["func"](geom["beta"])
    chi_mev = chi_scales[label]
    
    # 2. Calculate Physical Volume: V = T * L^3 * a^4
    V_phys = (geom["T"] * a_fm) * (geom["L"] * a_fm)**3
    
    # 3. Convert Chi to fm^-4: chi_fm = (chi_mev / 197.3)^4
    chi_fm4 = (chi_mev / HBAR_C)**4
    
    # 4. Calculate Expected <Q^2> and Spread
    expected_q2 = chi_fm4 * V_phys
    spread = np.sqrt(expected_q2)
    
    print(f"--- {label} (beta={geom['beta']}) ---")
    print(f"Lattice: {geom['T']} x {geom['L']}^3, a = {a_fm:.4f} fm")
    print(f"Physical Volume: {V_phys:.4f} fm^4")
    print(f"Expected <Q^2>:  {expected_q2:.2f}")
    print(f"Expected Spread: {spread:.2f} (Standard deviation of Q)")
    print()

--- SU(2) (beta=2.5) ---
Lattice: 20 x 16^3, a = 0.0774 fm
Physical Volume: 2.9433 fm^4
Expected <Q^2>:  3.11
Expected Spread: 1.76 (Standard deviation of Q)

--- SU(3) (beta=6.1) ---
Lattice: 20 x 16^3, a = 0.0789 fm
Physical Volume: 3.1762 fm^4
Expected <Q^2>:  2.79
Expected Spread: 1.67 (Standard deviation of Q)



In [8]:
beta = 3.0
N = 1

a_fm   = lattice_spacing_su2(beta)
X_ref  = X_ref_su2
HBAR_C = 200

chi_t_fm4 = (X_ref / HBAR_C)**4
V         = (N * a_fm)**4

spread = np.sqrt(chi_t_fm4 * V)
print(f"a = {a_fm:.4f} fm,  V = {V:.4f} fm⁴,  spread = {spread:.2f}")


NameError: name 'X_ref_su2' is not defined

In [ ]:
(0.0775 * (16))**8

5.589506702973337

In [ ]:
0.5**2

0.25